# UFC Fight Prediction — Data Exploration

This notebook explores the **bout_features** table produced by the Phase 3 feature pipeline. It covers:

1. **Dataset overview** — shape, date range, label distribution
2. **Missingness analysis** — which features are sparse and why
3. **Feature distributions** — key statistics and histograms
4. **Correlations** — feature-to-label and inter-feature relationships
5. **Temporal structure** — how the data splits for time-based modeling

**Prerequisites:** Run `make features_up` to populate the feature tables before using this notebook.

---

## 1. Setup

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add repo root to path so we can import project modules
sys.path.insert(0, str(Path.cwd().parent))

from warehouse.db import get_connection
from modeling.data import load_bout_data, temporal_split, rolling_cv, FEATURE_COLS, describe_splits

# Plot style
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:.4f}".format)

In [ ]:
conn = get_connection()
df = load_bout_data(conn)
conn.close()

print(f"Loaded {len(df):,} labeled bouts ({len(df.columns)} columns)")
print(f"Date range: {df['event_date'].min().date()} → {df['event_date'].max().date()}")
print(f"Feature columns: {len(FEATURE_COLS)}")
df.head()

## 2. Dataset Overview

### Label distribution

`label = 1` means fighter_1 won; `label = 0` means fighter_2 won. Draws and no-contests are excluded (NULL labels filtered by `load_bout_data`).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Overall label distribution
counts = df["label"].value_counts().sort_index()
axes[0].bar(["fighter_2 wins (0)", "fighter_1 wins (1)"], counts.values, color=["#e74c3c", "#2ecc71"])
axes[0].set_title("Overall Label Distribution")
axes[0].set_ylabel("Count")
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 30, f"{v:,} ({100*v/len(df):.1f}%)", ha="center", fontweight="bold")

# Label distribution over time (yearly)
df["year"] = df["event_date"].dt.year
yearly = df.groupby("year")["label"].agg(["mean", "count"])
axes[1].plot(yearly.index, yearly["mean"], "o-", color="#3498db", markersize=4)
axes[1].axhline(y=0.5, color="gray", linestyle="--", alpha=0.5)
axes[1].set_title("Fighter_1 Win Rate by Year")
axes[1].set_ylabel("P(label = 1)")
axes[1].set_xlabel("Year")
axes[1].set_ylim(0.3, 0.9)

plt.tight_layout()
plt.show()

print(f"\nOverall fighter_1 win rate: {df['label'].mean():.3f}")
print(f"Post-2015 fighter_1 win rate: {df.loc[df['year'] >= 2015, 'label'].mean():.3f}")

### Fights per year

Volume of fights has grown substantially since the early UFC days.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
fights_per_year = df.groupby("year").size()
ax.bar(fights_per_year.index, fights_per_year.values, color="#3498db", alpha=0.8)
ax.set_title("Number of Fights per Year")
ax.set_xlabel("Year")
ax.set_ylabel("Fights")
plt.tight_layout()
plt.show()

## 3. Missingness Analysis

Features can be NULL for legitimate reasons:
- **Debut fighters** have no career history → all diff/ratio features are NULL
- **Sparse bios** on ufcstats.com → age, height, reach missing for some fighters
- **No takedown attempts** → takedown accuracy is undefined

Understanding missingness is critical because LightGBM handles NaN natively, but logistic regression requires imputation.

In [ ]:
null_pct = df[FEATURE_COLS].isnull().mean().sort_values(ascending=False) * 100
null_pct = null_pct[null_pct > 0]

fig, ax = plt.subplots(figsize=(10, max(4, len(null_pct) * 0.3)))
null_pct.plot.barh(ax=ax, color="#e67e22")
ax.set_xlabel("% NULL")
ax.set_title("Feature Missingness (only features with any NULLs)")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print(f"\nFeatures with 0% missing: {(df[FEATURE_COLS].isnull().mean() == 0).sum()} / {len(FEATURE_COLS)}")
print(f"Both-debuting bouts (all diffs NULL): {(df['both_debuting'] == 1).sum()} ({100*(df['both_debuting'] == 1).mean():.1f}%)")

## 4. Feature Distributions

### Summary statistics

In [ ]:
df[FEATURE_COLS].describe().T

### Key feature histograms

Difference features (fighter_1 - fighter_2) should be roughly centered at zero if the matchmaking is balanced. Deviations from zero indicate systematic ordering of fighter_1 vs fighter_2.

In [ ]:
key_features = [
    "diff_elo", "diff_career_win_rate", "diff_career_fights",
    "diff_age", "diff_reach_cm", "diff_win_rate_decay",
    "ratio_elo", "ratio_career_wins",
    "diff_career_sig_strike_accuracy",
]

fig, axes = plt.subplots(3, 3, figsize=(14, 10))
for ax, col in zip(axes.flat, key_features):
    data = df[col].dropna()
    ax.hist(data, bins=50, color="#3498db", alpha=0.7, edgecolor="white")
    ax.set_title(col, fontsize=10)
    ax.axvline(x=0, color="red", linestyle="--", alpha=0.5)
    ax.tick_params(labelsize=8)

plt.suptitle("Key Feature Distributions", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 5. Correlations

### Feature-to-label correlation

Which features correlate most with the outcome? High correlations (|r| > 0.5) would be suspicious — potential leakage. Moderate correlations (0.1–0.3) are expected for useful predictors.

In [ ]:
# Numeric features only (drop booleans that are already int)
numeric_feats = [c for c in FEATURE_COLS if df[c].dtype in ("float64", "int64", "Int64")]
corr_with_label = df[numeric_feats].corrwith(df["label"]).sort_values()

fig, ax = plt.subplots(figsize=(10, max(4, len(corr_with_label) * 0.28)))
colors = ["#e74c3c" if v < 0 else "#2ecc71" for v in corr_with_label.values]
corr_with_label.plot.barh(ax=ax, color=colors)
ax.set_xlabel("Pearson r with label")
ax.set_title("Feature Correlation with Fight Outcome")
ax.axvline(x=0, color="black", linewidth=0.8)
ax.axvline(x=0.5, color="red", linestyle="--", alpha=0.3, label="|r| = 0.5 (suspicious)")
ax.axvline(x=-0.5, color="red", linestyle="--", alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Max |r|: {corr_with_label.abs().max():.3f} ({corr_with_label.abs().idxmax()})")
print("No features exceed |r| > 0.5 — no leakage signal." if corr_with_label.abs().max() < 0.5 
      else "WARNING: features with |r| > 0.5 detected!")

### Inter-feature correlation heatmap

Highly correlated feature pairs are redundant — tree models handle this fine, but logistic regression may be affected. Clusters of correlated features usually represent the same underlying signal measured different ways (e.g. career win rate, win rate last 3, and win rate decay are all "how much does this fighter win?").

In [ ]:
# Compute on non-boolean numeric features
corr_feats = [c for c in FEATURE_COLS 
              if c not in ("is_title_fight", "is_orthodox_vs_southpaw", "both_debuting")]
corr_matrix = df[corr_feats].corr()

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, center=0, cmap="RdBu_r", vmin=-1, vmax=1,
            square=True, linewidths=0.5, ax=ax, fmt=".1f",
            cbar_kws={"shrink": 0.8, "label": "Pearson r"})
ax.set_title("Feature-to-Feature Correlation Matrix", pad=20)
plt.tight_layout()
plt.show()

# Flag highly correlated pairs
high_corr = []
for i in range(len(corr_feats)):
    for j in range(i + 1, len(corr_feats)):
        r = corr_matrix.iloc[i, j]
        if abs(r) > 0.8:
            high_corr.append((corr_feats[i], corr_feats[j], r))

if high_corr:
    print(f"\nHighly correlated pairs (|r| > 0.8):")
    for a, b, r in sorted(high_corr, key=lambda x: -abs(x[2])):
        print(f"  {r:+.3f}  {a}  ↔  {b}")
else:
    print("\nNo feature pairs with |r| > 0.8")

## 6. Feature Distributions by Outcome

Do features look different for fighter_1 wins vs losses? Separable distributions suggest predictive power.

In [ ]:
top_corr_features = corr_with_label.abs().sort_values(ascending=False).head(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flat, top_corr_features):
    for label_val, color, name in [(1, "#2ecc71", "F1 wins"), (0, "#e74c3c", "F2 wins")]:
        data = df.loc[df["label"] == label_val, col].dropna()
        ax.hist(data, bins=40, alpha=0.5, color=color, label=name, density=True)
    ax.set_title(col, fontsize=10)
    ax.legend(fontsize=8)
    ax.tick_params(labelsize=8)

plt.suptitle("Top 6 Correlated Features — Distribution by Outcome", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 7. Temporal Splits

Time-based splitting is essential for fight prediction — you can never train on future fights. Here are the default splits and rolling CV folds.

### Train / Validation / Test split

In [ ]:
train, val, test = temporal_split(df)
describe_splits(train, val, test)

# Visualize splits on timeline
fig, ax = plt.subplots(figsize=(14, 3))
for split_df, color, label in [(train, "#3498db", "Train"), (val, "#f39c12", "Validation"), (test, "#e74c3c", "Test")]:
    dates = split_df["event_date"]
    ax.fill_between([dates.min(), dates.max()], 0, 1, alpha=0.3, color=color, label=label)
    ax.axvline(x=dates.min(), color=color, linestyle="--", alpha=0.7)
    ax.text(dates.min() + (dates.max() - dates.min()) / 2, 0.5,
            f"{label}\nn={len(split_df):,}", ha="center", va="center", fontsize=11, fontweight="bold")

ax.set_yticks([])
ax.set_title("Temporal Split Timeline")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

### Rolling CV folds

Expanding-window cross-validation: each fold trains on all history before its validation window. Later folds have more training data.

In [ ]:
folds = rolling_cv(df, n_folds=5, min_train_years=3, val_months=12)

fig, ax = plt.subplots(figsize=(14, max(3, len(folds) * 0.8 + 1)))
colors_train = plt.cm.Blues(np.linspace(0.3, 0.7, len(folds)))
colors_val = plt.cm.Oranges(np.linspace(0.4, 0.8, len(folds)))

for i, (tr, vl) in enumerate(folds):
    y = len(folds) - i
    # Training bar
    ax.barh(y, (tr["event_date"].max() - tr["event_date"].min()).days,
            left=tr["event_date"].min(), height=0.4, color=colors_train[i],
            label="Train" if i == 0 else None)
    # Validation bar
    ax.barh(y, (vl["event_date"].max() - vl["event_date"].min()).days,
            left=vl["event_date"].min(), height=0.4, color=colors_val[i],
            label="Val" if i == 0 else None)
    ax.text(vl["event_date"].max() + pd.Timedelta(days=30), y,
            f"train={len(tr):,}  val={len(vl):,}", va="center", fontsize=9)

ax.set_yticks(range(1, len(folds) + 1))
ax.set_yticklabels([f"Fold {len(folds) - i}" for i in range(len(folds))])
ax.set_title(f"Rolling CV ({len(folds)} folds, 12-month validation windows)")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 8. Elo as a Standalone Predictor

The Elo difference (`diff_elo`) can be converted directly to a win probability using the logistic formula. This is the simplest possible model — a useful baseline.

In [ ]:
# Elo → probability: P = 1 / (1 + 10^(-diff_elo / 400))
test_elo = test.copy()
test_elo["p_elo"] = 1 / (1 + 10 ** (-test_elo["diff_elo"] / 400))

# Bin by predicted probability and compute actual win rate
test_elo["prob_bin"] = pd.cut(test_elo["p_elo"], bins=np.arange(0.3, 0.75, 0.05))
calibration = test_elo.groupby("prob_bin", observed=True).agg(
    mean_predicted=("p_elo", "mean"),
    mean_actual=("label", "mean"),
    count=("label", "count"),
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Calibration plot
ax = axes[0]
ax.plot([0, 1], [0, 1], "k--", alpha=0.3, label="Perfect calibration")
ax.scatter(calibration["mean_predicted"], calibration["mean_actual"],
           s=calibration["count"] * 3, alpha=0.7, color="#3498db", edgecolors="white")
ax.set_xlabel("Predicted P(fighter_1 wins)")
ax.set_ylabel("Actual win rate")
ax.set_title("Elo Baseline — Calibration (Test Set)")
ax.legend()
ax.set_xlim(0.3, 0.7)
ax.set_ylim(0.3, 0.8)

# Accuracy by confidence
ax = axes[1]
test_elo["elo_pred"] = (test_elo["p_elo"] >= 0.5).astype(int)
test_elo["correct"] = test_elo["elo_pred"] == test_elo["label"]
test_elo["confidence"] = test_elo["p_elo"].apply(lambda p: max(p, 1 - p))
test_elo["conf_bin"] = pd.cut(test_elo["confidence"], bins=[0.5, 0.52, 0.54, 0.56, 0.58, 0.60, 0.65, 1.0])
conf_acc = test_elo.groupby("conf_bin", observed=True).agg(
    accuracy=("correct", "mean"), count=("correct", "count"))
ax.bar(range(len(conf_acc)), conf_acc["accuracy"], color="#2ecc71", alpha=0.7)
ax.set_xticks(range(len(conf_acc)))
ax.set_xticklabels([str(b) for b in conf_acc.index], rotation=45, fontsize=8)
ax.set_ylabel("Accuracy")
ax.set_title("Elo Accuracy by Confidence Bucket")
ax.axhline(y=0.5, color="gray", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

elo_acc = test_elo["correct"].mean()
print(f"Elo baseline accuracy on test set: {elo_acc:.3f}")
print(f"(A coin flip is 0.500 — any model must beat this and ideally beat Elo)")

## 9. Takeaways for Modeling

Key observations from this exploration:

1. **Label imbalance is era-dependent.** Early UFC had ~80% fighter_1 win rate (champion listed first). Modern data is closer to 55%. Time-based splits naturally handle this — the test set reflects current conditions.

2. **Missingness is structural, not random.** It comes from debut fighters (no history) and sparse bios. LightGBM handles this natively; logistic regression needs imputation.

3. **No leakage signal.** No feature correlates with the label above |r| = 0.5. The strongest predictor is `diff_age` (r ~ -0.19) — younger fighters tend to win.

4. **Feature redundancy exists.** Career win rate, rolling win rate, and decayed win rate are correlated (~0.7–0.9). Tree models absorb this; logistic regression may want feature selection.

5. **Elo is a reasonable baseline.** Even a simple Elo → probability conversion gives a starting accuracy benchmark that any model must beat.

6. **~8,400 labeled rows** is a small dataset by ML standards. Overfitting is a real risk — conservative hyperparameters and proper temporal validation are essential.

---

*Next: `02_model_comparison.ipynb` — train baselines and models, compare calibration and metrics side-by-side.*